In [1]:
import os

FEATURES = ["P_dyn", "VBs", "epsilon", "DST"]
MODE = "template"  # 'template' or 'default'
OUTPUT_DIR = "comparison_template_deriv_against_baselines"
RAW_EQ = "g = (#2 * -0.0010713526) * sqrt(#1 + 1.32442); d = square((#1 * 0.01838707) - 0.5912093)"

output_folder = OUTPUT_DIR
# Count number of existing subfolders
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

import sympy as sp
from tqdm import tqdm

from sympy.printing import latex

# Internal module imports
import storm_dates
import baseline_models

from evaluation_engine import EquationModel, simulate_storm
from train_script import load_and_preprocess, compute_features

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


/mnt/data/symbolic-regression-dst-public-repo/.venv/lib/python3.12/site-packages/spacepy/time.py:2448: UserWarning: Leapseconds may be out of date. Use spacepy.toolbox.update(leapsecs=True)
  _read_leaps()


In [3]:
raw_data = load_and_preprocess()
data = compute_features(raw_data)

In [4]:
def predict_and_plot_storm(model, start, end, storm_df, storm_id, save_path):
    # 1. Generate Predictions
    y_true = storm_df[start:end]["DST"].values

    res_eq = simulate_storm(model, storm_df)
    res_burton = baseline_models.burton_prediction(storm_df)
    res_obm = baseline_models.obm_prediction(storm_df)
    ddm1 = baseline_models.ddm1_prediction(storm_df)
    ddm2 = baseline_models.ddm2_prediction(storm_df)
    ddm3 = baseline_models.ddm3_prediction(storm_df)

    res_eq = res_eq[start:end]["DST_pred"].values
    res_burton = res_burton[start:end]["DST_pred"].values
    res_obm = res_obm[start:end]["DST_pred"].values
    res_ddm1 = ddm1[start:end]["DST_pred"].values
    res_ddm2 = ddm2[start:end]["DST_pred"].values
    res_ddm3 = ddm3[start:end]["DST_pred"].values

    # 2. Calculate Metrics
    m_eq = baseline_models.get_all_metrics(y_true, res_eq, get_cc=False)
    m_burton = baseline_models.get_all_metrics(y_true, res_burton, get_cc=False)
    m_obm = baseline_models.get_all_metrics(y_true, res_obm, get_cc=False)
    m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1, get_cc=False)
    m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2, get_cc=False)
    m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3, get_cc=False)

    # 3. Setup Figure (3 Columns)
    fig, axs = plt.subplots(1, 3, figsize=(24, 11), constrained_layout=True)
    fig.suptitle(
        rf"Evaluation for Equation: ${model.latex_str()}$", fontsize=24, wrap=True
    )
    # Column 1: Time Series
    axs[0].plot(
        storm_df[start:end].index,
        y_true,
        color="black",
        label="Observed",
        alpha=0.6,
        linewidth=2,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_eq,
        color="blue",
        linestyle="--",
        label="Equation",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_burton,
        color="yellow",
        linestyle="--",
        label="Burton",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_obm,
        color="green",
        linestyle="--",
        label="OBM",
        linewidth=1.5,
    )
    
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm1,
        color="orange",
        linestyle="--",
        label="DDM1",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm2,
        color="purple",
        linestyle="--",
        label="DDM2",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm3,
        color="cyan",
        linestyle="--",
        label="DDM3",
        linewidth=1.5,
    )
    
    axs[0].set_title(f"Storm {storm_id} Reconstruction", fontsize=24)
    axs[0].tick_params(axis='both', which='major', labelsize=20)
    axs[0].tick_params(axis='both', which='minor', labelsize=18)
    axs[0].legend(fontsize = 18)
    axs[0].grid(True)
    axs[0].set_xlim(start, end)
    axs[0].set_xlabel("Date", fontsize=20)
    axs[0].set_ylabel("Dst (nT)", fontsize=20)
    axs[0].xaxis.set_major_locator(MultipleLocator(2))

    # Column 2: Prediction Error
    diff_eq = res_eq - y_true
    diff_burton = res_burton - y_true
    diff_obm = res_obm - y_true
    diff_ddm1 = res_ddm1 - y_true
    diff_ddm2 = res_ddm2 - y_true
    diff_ddm3 = res_ddm3 - y_true


    axs[1].plot(storm_df[start:end].index, diff_eq, color="blue", label="Eq Error")
    axs[1].plot(
        storm_df[start:end].index, diff_burton, color="yellow", label="Burton Error"
    )
    axs[1].plot(storm_df[start:end].index, diff_obm, color="green", label="OBM Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm1, color="orange", label="DDM1 Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm2, color="purple", label="DDM2 Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm3, color="cyan", label="DDM3 Error")
    axs[1].axhline(0, color="black", linestyle="--")
    axs[1].set_title("Prediction Error", fontsize=24)
    axs[1].set_ylabel("Error (nT)", fontsize=20)
    axs[1].set_xlabel("Date", fontsize=20)

    title_metrics = (
        f"Error Comparison\n"
        f"Eq: RMSE {m_eq[0]:.2f} | MAE {m_eq[1]:.2f} | R2 {m_eq[2]:.2f} | BFE {m_eq[3]:.2f}\n"
        f"Burton: RMSE {m_burton[0]:.2f} | MAE {m_burton[1]:.2f} | R2 {m_burton[2]:.2f} | BFE {m_burton[3]:.2f}\n"
        f"OBM: RMSE {m_obm[0]:.2f} | MAE {m_obm[1]:.2f} | R2 {m_obm[2]:.2f} | BFE {m_obm[3]:.2f}\n"
        f"DDM1: RMSE {m_ddm1[0]:.2f} | MAE {m_ddm1[1]:.2f} | R2 {m_ddm1[2]:.2f} | BFE {m_ddm1[3]:.2f}\n"
        f"DDM2: RMSE {m_ddm2[0]:.2f} | MAE {m_ddm2[1]:.2f} | R2 {m_ddm2[2]:.2f} | BFE {m_ddm2[3]:.2f}\n"
        f"DDM3: RMSE {m_ddm3[0]:.2f} | MAE {m_ddm3[1]:.2f} | R2 {m_ddm3[2]:.2f} | BFE {m_ddm3[3]:.2f}"
    )
    axs[1].set_title(title_metrics, fontsize=24)
    axs[1].set_ylabel("Error (nT)", fontsize=20)
    axs[1].set_xlabel("Date", fontsize=20)
    axs[1].legend(fontsize=18)
    axs[1].grid(True)
    axs[1].set_xlim(start, end)
    
    axs[1].tick_params(axis="both", which="major", labelsize=20)
    axs[1].tick_params(axis="both", which="minor", labelsize=18)
        
    axs[1].xaxis.set_major_locator(MultipleLocator(2))

    if len(axs[1].xaxis.get_ticklabels()) > 6:    
        for label in axs[1].xaxis.get_ticklabels()[1::2]:
            label.set_visible(False)

    # Column 3: BFE
    baseline_models.plot_evaluation_bfe_multi(
        axs[2],
        y_true,
        [res_eq, res_burton, res_obm, res_ddm1, res_ddm2, res_ddm3],
        ["Equation", "Burton", "OBM", "DDM1", "DDM2", "DDM3"],
        ["blue", "yellow", "green", "orange", "purple", "cyan"],
        fontsize = 20,
        plot_legend=True
    )

    plt.savefig(save_path)
    plt.close()


In [5]:
def save_prediction_data(model, start, end, storm_df, output_path):
    """
    Generates and saves a CSV with observed and predicted DST and dDST/dt.
    """
    # 1. Observed Data
    # Real dDST is calculated as the difference to the next hour
    real_dst = storm_df[start:end]["DST"].values
    real_ddst = storm_df[start:end]["DST"].diff().shift(-1).values

    # 2. Equation Predictions
    # We need the iterative predictions for DST
    pred_dst_eq = simulate_storm(model, storm_df)
    if model.is_template:
        pred_dst_eq = pred_dst_eq[start:end][
            ["DST_pred", "dDST", "injection_component", "decay_component"]
        ]
    else:
        pred_dst_eq = pred_dst_eq[start:end][["DST_pred", "dDST"]]
    # 3. Baseline Predictions (Burton & OBM)
    pred_dst_burton = baseline_models.burton_prediction(storm_df)
    pred_dst_burton = pred_dst_burton[start:end][["DST_pred", "dDST"]]
    pred_dst_burton.columns = ["DST_pred_burton", "dDST_burton"]
    pred_dst_burton = pred_dst_burton[start:end][["DST_pred_burton", "dDST_burton"]]
    pred_dst_obm = baseline_models.obm_prediction(storm_df)
    pred_dst_obm = pred_dst_obm[start:end][["DST_pred", "dDST"]]
    pred_dst_obm.columns = ["DST_pred_obm", "dDST_obm"]
    pred_dst_obm = pred_dst_obm[start:end][["DST_pred_obm", "dDST_obm"]]
    pred_dst_ddm1 = baseline_models.ddm1_prediction(storm_df)    
    pred_dst_ddm1.columns = ["DST_pred_ddm1", "dDST_ddm1"]
    pred_dst_ddm1 = pred_dst_ddm1[start:end][["DST_pred_ddm1", "dDST_ddm1"]]
    pred_dst_ddm2 = baseline_models.ddm2_prediction(storm_df)
    pred_dst_ddm2.columns = ["DST_pred_ddm2", "dDST_ddm2"]
    pred_dst_ddm2 = pred_dst_ddm2[start:end][["DST_pred_ddm2", "dDST_ddm2"]]
    pred_dst_ddm3 = baseline_models.ddm3_prediction(storm_df)
    pred_dst_ddm3.columns = ["DST_pred_ddm3", "dDST_ddm3"]
    pred_dst_ddm3 = pred_dst_ddm3[start:end][["DST_pred_ddm3", "dDST_ddm3"]]


    # 4. Construct Comprehensive DataFrame

    if model.is_template:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,
                "Pred_DST_Equation": pred_dst_eq["DST_pred"].values,
                "Pred_dDST_dt_Equation": pred_dst_eq["dDST"].values,
                "Injection_Component": pred_dst_eq["injection_component"].values,
                "Decay_Component": pred_dst_eq["decay_component"].values,
                "Pred_DST_Burton": pred_dst_burton["DST_pred_burton"].values,
                "Pred_dDST_dt_Burton": pred_dst_burton["dDST_burton"].values,
                "Pred_DST_OBM": pred_dst_obm["DST_pred_obm"].values,
                "Pred_dDST_dt_OBM": pred_dst_obm["dDST_obm"].values,
                "Pred_DST_DDM1": pred_dst_ddm1["DST_pred_ddm1"].values,
                "Pred_dDST_dt_DDM1": pred_dst_ddm1["dDST_ddm1"].values,
                "Pred_DST_DDM2": pred_dst_ddm2["DST_pred_ddm2"].values,
                "Pred_dDST_dt_DDM2": pred_dst_ddm2["dDST_ddm2"].values,
                "Pred_DST_DDM3": pred_dst_ddm3["DST_pred_ddm3"].values,
                "Pred_dDST_dt_DDM3": pred_dst_ddm3["dDST_ddm3"].values,
            }
        ).set_index("Timestamp")
    else:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,
                "Pred_DST_Equation": pred_dst_eq["DST_pred"].values,
                "Pred_dDST_dt_Equation": pred_dst_eq["dDST"].values,
                "Pred_DST_Burton": pred_dst_burton["DST_pred_burton"].values,
                "Pred_dDST_dt_Burton": pred_dst_burton["dDST_burton"].values,
                "Pred_DST_OBM": pred_dst_obm["DST_pred_obm"].values,
                "Pred_dDST_dt_OBM": pred_dst_obm["dDST_obm"].values,
                "Pred_DST_DDM1": pred_dst_ddm1["DST_pred_ddm1"].values,
                "Pred_dDST_dt_DDM1": pred_dst_ddm1["dDST_ddm1"].values,
                "Pred_DST_DDM2": pred_dst_ddm2["DST_pred_ddm2"].values,
                "Pred_dDST_dt_DDM2": pred_dst_ddm2["dDST_ddm2"].values,
                "Pred_DST_DDM3": pred_dst_ddm3["DST_pred_ddm3"].values,
                "Pred_dDST_dt_DDM3": pred_dst_ddm3["dDST_ddm3"].values,

            }
        ).set_index("Timestamp")

    results_df.to_csv(output_path)
    return results_df

## Test storms

In [6]:
storms = []

model = EquationModel(RAW_EQ, FEATURES, is_template=MODE == "template")
test_storms = storm_dates.TEST_STORMS_SYMBOLIC_REGRESSION
storm_indices = []
for sd, ed, storm_id in tqdm(test_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        model,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms.append(
        save_prediction_data(
            model, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name),
        )
    )
    storm_indices.append(storm_id)
    
with open(os.path.join(OUTPUT_DIR, 'equation.txt'), 'w') as f:
    f.write(f'Equation: {RAW_EQ}\n')            
    f.write(f'LaTeX: {latex(model.latex_str())}\n')

  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 20/20 [00:12<00:00,  1.66it/s]


In [7]:
metrics = ["RMSE", "MAE", "R2", "BFE"]
equations = ["Equation", "Burton", "OBM", "DDM1", "DDM2", "DDM3"]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms):
    y_true = storm["Observed_DST"].values
    res_eq = storm["Pred_DST_Equation"].values
    res_burton = storm["Pred_DST_Burton"].values
    res_obm = storm["Pred_DST_OBM"].values
    res_ddm1 = storm["Pred_DST_DDM1"].values
    res_ddm2 = storm["Pred_DST_DDM2"].values
    res_ddm3 = storm["Pred_DST_DDM3"].values

    m_eq = baseline_models.get_all_metrics(y_true, res_eq, get_cc = False)
    m_burton = baseline_models.get_all_metrics(y_true, res_burton, get_cc = False)
    m_obm = baseline_models.get_all_metrics(y_true, res_obm, get_cc = False)
    m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1, get_cc = False)
    m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2, get_cc = False)
    m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3, get_cc = False)

    summary_df.loc[len(summary_df)] = [
        storm_indices[storm_index],
        *m_eq,
        *m_burton,
        *m_obm,
        *m_ddm1,
        *m_ddm2,
        *m_ddm3,
    ]

summary_df.loc[len(summary_df)] = ["Mean", *summary_df[columns].mean().values]

global_data = pd.concat(storms, ignore_index=True)
y_true = global_data["Observed_DST"].values
res_eq = global_data["Pred_DST_Equation"].values
res_burton = global_data["Pred_DST_Burton"].values
res_obm = global_data["Pred_DST_OBM"].values
res_ddm1 = global_data["Pred_DST_DDM1"].values
res_ddm2 = global_data["Pred_DST_DDM2"].values
res_ddm3 = global_data["Pred_DST_DDM3"].values

m_eq = baseline_models.get_all_metrics(y_true, res_eq, get_cc = False)
m_burton = baseline_models.get_all_metrics(y_true, res_burton, get_cc = False)
m_obm = baseline_models.get_all_metrics(y_true, res_obm, get_cc = False)
m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1, get_cc = False)
m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2, get_cc = False)
m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3, get_cc = False)

summary_df.loc[len(summary_df)] = [
    "Global",
    *m_eq,
    *m_burton,
    *m_obm,
    *m_ddm1,
    *m_ddm2,
    *m_ddm3,
]

display(summary_df)

,Storm Index,Equation_RMSE,Equation_MAE,Equation_R2,Equation_BFE,Burton_RMSE,Burton_MAE,Burton_R2,Burton_BFE,OBM_RMSE,...,DDM1_R2,DDM1_BFE,DDM2_RMSE,DDM2_MAE,DDM2_R2,DDM2_BFE,DDM3_RMSE,DDM3_MAE,DDM3_R2,DDM3_BFE
0,54.0,9.082450,7.150428,0.796405,11.620567,22.373695,19.269475,-0.235483,22.860411,12.934864,...,0.228072,21.129540,17.034353,14.718489,0.283836,19.285046,17.429228,14.902320,0.250248,20.944809
1,55.0,17.429458,13.783357,0.748936,19.558619,22.579090,19.588765,0.578663,19.865986,17.163269,...,0.610980,23.435284,20.994874,17.672795,0.635713,22.088199,22.442717,18.173790,0.583737,25.386413
2,56.0,13.276276,10.962537,0.613082,15.586261,14.341538,9.959761,0.548500,20.706592,13.462829,...,0.590821,13.923395,11.710892,9.329760,0.698945,13.104050,11.378825,9.160091,0.715776,12.828221
3,57.0,9.579700,7.665862,0.766690,12.124595,19.246741,14.483019,0.058231,15.793575,10.406807,...,0.804520,13.120449,8.303250,6.470139,0.824723,11.441876,8.670891,6.332414,0.808857,13.883947
4,58.0,9.165543,6.390252,0.797016,15.584912,9.600592,7.210339,0.777290,11.977608,10.384637,...,0.637669,20.108684,11.990044,9.586551,0.652635,19.031650,11.862568,8.934847,0.659982,19.970899
5,59.0,12.207318,10.241290,0.849938,9.432197,16.064189,12.178801,0.740135,16.351529,16.381482,...,0.869938,14.982619,10.840170,8.734074,0.881668,13.497036,12.354960,9.578556,0.846286,16.759531
6,60.0,17.617883,11.828793,0.777722,26.807645,25.566310,20.034542,0.531914,33.653076,19.034334,...,0.650269,36.468144,13.320289,10.330457,0.872938,17.936865,14.717364,11.742180,0.844887,19.772733
7,61.0,10.679534,8.252146,0.932006,14.553100,40.570432,21.185698,0.018733,59.661410,18.231207,...,0.712652,28.911696,22.490189,16.826801,0.698454,30.616512,20.468681,15.214925,0.750226,30.187252
8,62.0,11.394979,9.122644,0.698127,12.820999,14.359693,10.613704,0.520612,13.397201,16.052525,...,0.760496,14.844234,10.405858,7.936314,0.748260,14.058279,11.805307,9.352887,0.675995,16.881751
9,63.0,14.975924,12.167919,0.837054,18.193426,22.708494,18.800912,0.625343,24.687730,16.107816,...,0.676623,29.948383,20.014049,16.071049,0.708977,28.117249,22.085126,17.377477,0.645630,32.083276


In [8]:
summary_df

,Storm Index,Equation_RMSE,Equation_MAE,Equation_R2,Equation_BFE,Burton_RMSE,Burton_MAE,Burton_R2,Burton_BFE,OBM_RMSE,...,DDM1_R2,DDM1_BFE,DDM2_RMSE,DDM2_MAE,DDM2_R2,DDM2_BFE,DDM3_RMSE,DDM3_MAE,DDM3_R2,DDM3_BFE
0,54.0,9.082450,7.150428,0.796405,11.620567,22.373695,19.269475,-0.235483,22.860411,12.934864,...,0.228072,21.129540,17.034353,14.718489,0.283836,19.285046,17.429228,14.902320,0.250248,20.944809
1,55.0,17.429458,13.783357,0.748936,19.558619,22.579090,19.588765,0.578663,19.865986,17.163269,...,0.610980,23.435284,20.994874,17.672795,0.635713,22.088199,22.442717,18.173790,0.583737,25.386413
2,56.0,13.276276,10.962537,0.613082,15.586261,14.341538,9.959761,0.548500,20.706592,13.462829,...,0.590821,13.923395,11.710892,9.329760,0.698945,13.104050,11.378825,9.160091,0.715776,12.828221
3,57.0,9.579700,7.665862,0.766690,12.124595,19.246741,14.483019,0.058231,15.793575,10.406807,...,0.804520,13.120449,8.303250,6.470139,0.824723,11.441876,8.670891,6.332414,0.808857,13.883947
4,58.0,9.165543,6.390252,0.797016,15.584912,9.600592,7.210339,0.777290,11.977608,10.384637,...,0.637669,20.108684,11.990044,9.586551,0.652635,19.031650,11.862568,8.934847,0.659982,19.970899
5,59.0,12.207318,10.241290,0.849938,9.432197,16.064189,12.178801,0.740135,16.351529,16.381482,...,0.869938,14.982619,10.840170,8.734074,0.881668,13.497036,12.354960,9.578556,0.846286,16.759531
6,60.0,17.617883,11.828793,0.777722,26.807645,25.566310,20.034542,0.531914,33.653076,19.034334,...,0.650269,36.468144,13.320289,10.330457,0.872938,17.936865,14.717364,11.742180,0.844887,19.772733
7,61.0,10.679534,8.252146,0.932006,14.553100,40.570432,21.185698,0.018733,59.661410,18.231207,...,0.712652,28.911696,22.490189,16.826801,0.698454,30.616512,20.468681,15.214925,0.750226,30.187252
8,62.0,11.394979,9.122644,0.698127,12.820999,14.359693,10.613704,0.520612,13.397201,16.052525,...,0.760496,14.844234,10.405858,7.936314,0.748260,14.058279,11.805307,9.352887,0.675995,16.881751
9,63.0,14.975924,12.167919,0.837054,18.193426,22.708494,18.800912,0.625343,24.687730,16.107816,...,0.676623,29.948383,20.014049,16.071049,0.708977,28.117249,22.085126,17.377477,0.645630,32.083276


In [9]:
display(summary_df.iloc[-2:, :].style.format({col: "{:.2f}" for col in columns}))

,Storm Index,Equation_RMSE,Equation_MAE,Equation_R2,Equation_BFE,Burton_RMSE,Burton_MAE,Burton_R2,Burton_BFE,OBM_RMSE,OBM_MAE,OBM_R2,OBM_BFE,DDM1_RMSE,DDM1_MAE,DDM1_R2,DDM1_BFE,DDM2_RMSE,DDM2_MAE,DDM2_R2,DDM2_BFE,DDM3_RMSE,DDM3_MAE,DDM3_R2,DDM3_BFE
20,Mean,13.82,10.56,0.79,16.38,26.14,18.34,0.43,29.43,16.91,13.13,0.72,18.10,18.75,14.59,0.66,23.89,18.20,14.04,0.69,22.84,18.66,14.10,0.68,24.26
21,Global,14.01,10.25,0.89,28.79,28.71,16.83,0.52,78.08,17.34,12.71,0.82,44.10,19.86,13.52,0.77,38.89,19.06,13.16,0.79,45.26,19.55,13.36,0.78,46.38


In [10]:
display(summary_df[['Storm Index', 'Equation_BFE', 'Burton_BFE', 'OBM_BFE', 'DDM1_BFE', 'DDM2_BFE', 'DDM3_BFE']])

,Storm Index,Equation_BFE,Burton_BFE,OBM_BFE,DDM1_BFE,DDM2_BFE,DDM3_BFE
0,54.0,11.620567,22.860411,15.494723,21.129540,19.285046,20.944809
1,55.0,19.558619,19.865986,17.012856,23.435284,22.088199,25.386413
2,56.0,15.586261,20.706592,17.920288,13.923395,13.104050,12.828221
3,57.0,12.124595,15.793575,10.567192,13.120449,11.441876,13.883947
4,58.0,15.584912,11.977608,13.089744,20.108684,19.031650,19.970899
5,59.0,9.432197,16.351529,10.467582,14.982619,13.497036,16.759531
6,60.0,26.807645,33.653076,15.764727,36.468144,17.936865,19.772733
7,61.0,14.553100,59.661410,24.712335,28.911696,30.616512,30.187252
8,62.0,12.820999,13.397201,13.831007,14.844234,14.058279,16.881751
9,63.0,18.193426,24.687730,21.265740,29.948383,28.117249,32.083276


In [11]:
print(summary_df.loc[summary_df["Storm Index"].isin([57, 68, 'Mean'])].to_latex(index=False, float_format="%.3f"))

\begin{tabular}{lrrrrrrrrrrrrrrrrrrrrrrrr}
\toprule
Storm Index & Equation_RMSE & Equation_MAE & Equation_R2 & Equation_BFE & Burton_RMSE & Burton_MAE & Burton_R2 & Burton_BFE & OBM_RMSE & OBM_MAE & OBM_R2 & OBM_BFE & DDM1_RMSE & DDM1_MAE & DDM1_R2 & DDM1_BFE & DDM2_RMSE & DDM2_MAE & DDM2_R2 & DDM2_BFE & DDM3_RMSE & DDM3_MAE & DDM3_R2 & DDM3_BFE \\
\midrule
57.000 & 9.580 & 7.666 & 0.767 & 12.125 & 19.247 & 14.483 & 0.058 & 15.794 & 10.407 & 8.641 & 0.725 & 10.567 & 8.769 & 6.755 & 0.805 & 13.120 & 8.303 & 6.470 & 0.825 & 11.442 & 8.671 & 6.332 & 0.809 & 13.884 \\
68.000 & 27.918 & 20.659 & 0.946 & 30.007 & 72.495 & 53.764 & 0.636 & 76.973 & 44.485 & 32.605 & 0.863 & 50.547 & 26.232 & 19.685 & 0.952 & 29.756 & 41.573 & 29.791 & 0.880 & 48.472 & 50.339 & 34.755 & 0.824 & 54.647 \\
Mean & 13.818 & 10.560 & 0.788 & 16.383 & 26.144 & 18.335 & 0.426 & 29.429 & 16.910 & 13.132 & 0.718 & 18.104 & 18.745 & 14.586 & 0.660 & 23.894 & 18.198 & 14.044 & 0.693 & 22.838 & 18.664 & 14.101 & 0.683 & 2

In [12]:
print(summary_df)

   Storm Index  Equation_RMSE  Equation_MAE  Equation_R2  Equation_BFE  \
0         54.0       9.082450      7.150428     0.796405     11.620567   
1         55.0      17.429458     13.783357     0.748936     19.558619   
2         56.0      13.276276     10.962537     0.613082     15.586261   
3         57.0       9.579700      7.665862     0.766690     12.124595   
4         58.0       9.165543      6.390252     0.797016     15.584912   
5         59.0      12.207318     10.241290     0.849938      9.432197   
6         60.0      17.617883     11.828793     0.777722     26.807645   
7         61.0      10.679534      8.252146     0.932006     14.553100   
8         62.0      11.394979      9.122644     0.698127     12.820999   
9         63.0      14.975924     12.167919     0.837054     18.193426   
10        64.0      10.130700      8.022478     0.802369     11.183879   
11        65.0      14.249835     10.390993     0.633954     16.794854   
12        66.0      18.264381     15.4

In [13]:
display(summary_df[['Storm Index', 'Equation_RMSE',  'Equation_MAE',  'Equation_R2',  'Equation_BFE']])

,Storm Index,Equation_RMSE,Equation_MAE,Equation_R2,Equation_BFE
0,54.0,9.082450,7.150428,0.796405,11.620567
1,55.0,17.429458,13.783357,0.748936,19.558619
2,56.0,13.276276,10.962537,0.613082,15.586261
3,57.0,9.579700,7.665862,0.766690,12.124595
4,58.0,9.165543,6.390252,0.797016,15.584912
5,59.0,12.207318,10.241290,0.849938,9.432197
6,60.0,17.617883,11.828793,0.777722,26.807645
7,61.0,10.679534,8.252146,0.932006,14.553100
8,62.0,11.394979,9.122644,0.698127,12.820999
9,63.0,14.975924,12.167919,0.837054,18.193426


In [14]:
df = summary_df.copy()

In [15]:
# Keep only the 20 individual storms
storm_df = df[
    ~df["Storm Index"].isin(["Mean", "Global"])
].copy()

baselines = {
    "Burton": "Burton",
    "OBM": "OBM",
    "DDM#1": "DDM1",
    "DDM#2": "DDM2",
    "DDM#3": "DDM3",
}

metrics = {
    "RMSE": "lower",
    "MAE": "lower",
    "R2": "higher",
    "BFE": "lower",
}

rows = []

for metric, direction in metrics.items():

    row = {
        "Metric": metric,
        "Equation": storm_df[f"Equation_{metric}"].mean(),
    }

    for baseline_name, prefix in baselines.items():

        eq = storm_df[f"Equation_{metric}"]
        baseline = storm_df[f"{prefix}_{metric}"]

        row[baseline_name] = baseline.mean()

        if direction == "lower":
            wins = (eq < baseline).sum()
        else:
            wins = (eq > baseline).sum()

        row[f"Wins vs {baseline_name}"] = wins

    rows.append(row)

results_df = pd.DataFrame(rows)

print(results_df)

  Metric   Equation     Burton  Wins vs Burton        OBM  Wins vs OBM  \
0   RMSE  13.818406  26.143515              20  16.910134           17   
1    MAE  10.560487  18.335220              19  13.132016           17   
2     R2   0.788368   0.426411              20   0.717826           17   
3    BFE  16.382973  29.429250              18  18.104329           12   

       DDM#1  Wins vs DDM#1      DDM#2  Wins vs DDM#2      DDM#3  \
0  18.745373             15  18.198309             13  18.664223   
1  14.585654             14  14.043855             14  14.100546   
2   0.660410             15   0.692687             13   0.682677   
3  23.893669             18  22.837899             16  24.258579   

   Wins vs DDM#3  
0             16  
1             14  
2             16  
3             18  


In [16]:
latex_table = results_df.to_latex(
    index=False,
    escape=False,
    float_format="%.2f"
)

print(latex_table)

\begin{tabular}{lrrrrrrrrrrr}
\toprule
Metric & Equation & Burton & Wins vs Burton & OBM & Wins vs OBM & DDM#1 & Wins vs DDM#1 & DDM#2 & Wins vs DDM#2 & DDM#3 & Wins vs DDM#3 \\
\midrule
RMSE & 13.82 & 26.14 & 20 & 16.91 & 17 & 18.75 & 15 & 18.20 & 13 & 18.66 & 16 \\
MAE & 10.56 & 18.34 & 19 & 13.13 & 17 & 14.59 & 14 & 14.04 & 14 & 14.10 & 14 \\
R2 & 0.79 & 0.43 & 20 & 0.72 & 17 & 0.66 & 15 & 0.69 & 13 & 0.68 & 16 \\
BFE & 16.38 & 29.43 & 18 & 18.10 & 12 & 23.89 & 18 & 22.84 & 16 & 24.26 & 18 \\
\bottomrule
\end{tabular}



In [17]:
results_df

,Metric,Equation,Burton,Wins vs Burton,OBM,Wins vs OBM,DDM#1,Wins vs DDM#1,DDM#2,Wins vs DDM#2,DDM#3,Wins vs DDM#3
0,RMSE,13.818406,26.143515,20,16.910134,17,18.745373,15,18.198309,13,18.664223,16
1,MAE,10.560487,18.335220,19,13.132016,17,14.585654,14,14.043855,14,14.100546,14
2,R2,0.788368,0.426411,20,0.717826,17,0.660410,15,0.692687,13,0.682677,16
3,BFE,16.382973,29.429250,18,18.104329,12,23.893669,18,22.837899,16,24.258579,18


In [18]:
print(summary_df.drop(columns=summary_df.columns[1:5]).to_latex(index=False, float_format="%.3f").replace("_", " ").replace("Equation ", "Eq. 21 ").replace("Storm Index", "Storm"))

\begin{tabular}{lrrrrrrrrrrrrrrrrrrrr}
\toprule
Storm & Burton RMSE & Burton MAE & Burton R2 & Burton BFE & OBM RMSE & OBM MAE & OBM R2 & OBM BFE & DDM1 RMSE & DDM1 MAE & DDM1 R2 & DDM1 BFE & DDM2 RMSE & DDM2 MAE & DDM2 R2 & DDM2 BFE & DDM3 RMSE & DDM3 MAE & DDM3 R2 & DDM3 BFE \\
\midrule
54.000 & 22.374 & 19.269 & -0.235 & 22.860 & 12.935 & 10.340 & 0.587 & 15.495 & 17.685 & 15.097 & 0.228 & 21.130 & 17.034 & 14.718 & 0.284 & 19.285 & 17.429 & 14.902 & 0.250 & 20.945 \\
55.000 & 22.579 & 19.589 & 0.579 & 19.866 & 17.163 & 14.453 & 0.757 & 17.013 & 21.696 & 17.787 & 0.611 & 23.435 & 20.995 & 17.673 & 0.636 & 22.088 & 22.443 & 18.174 & 0.584 & 25.386 \\
56.000 & 14.342 & 9.960 & 0.549 & 20.707 & 13.463 & 9.386 & 0.602 & 17.920 & 13.653 & 10.595 & 0.591 & 13.923 & 11.711 & 9.330 & 0.699 & 13.104 & 11.379 & 9.160 & 0.716 & 12.828 \\
57.000 & 19.247 & 14.483 & 0.058 & 15.794 & 10.407 & 8.641 & 0.725 & 10.567 & 8.769 & 6.755 & 0.805 & 13.120 & 8.303 & 6.470 & 0.825 & 11.442 & 8.671 & 6.332 

## Train storms

In [19]:
storms_train = []

train_storms = storm_dates.TRAIN_STORMS_SYMBOLIC_REGRESSION
storm_indices_train = []
for sd, ed, storm_id in tqdm(train_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        print(f'Storm {storm_id} has no data from {start} to {end}. Skipping.')
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        model,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms_train.append(
        save_prediction_data(
            model, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    storm_indices_train.append(storm_id)

100%|██████████| 53/53 [00:31<00:00,  1.67it/s]


In [20]:
metrics = ["RMSE", "MAE", "R2", "BFE"]
equations = ["Equation", "Burton", "OBM", "DDM1", "DDM2", "DDM3"]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df_train = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms_train):
    y_true = storm["Observed_DST"].values
    res_eq = storm["Pred_DST_Equation"].values
    res_burton = storm["Pred_DST_Burton"].values
    res_obm = storm["Pred_DST_OBM"].values
    res_ddm1 = storm["Pred_DST_DDM1"].values
    res_ddm2 = storm["Pred_DST_DDM2"].values
    res_ddm3 = storm["Pred_DST_DDM3"].values

    m_eq = baseline_models.get_all_metrics(y_true, res_eq, get_cc = False)
    m_burton = baseline_models.get_all_metrics(y_true, res_burton, get_cc = False)
    m_obm = baseline_models.get_all_metrics(y_true, res_obm, get_cc = False)
    m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1, get_cc = False)
    m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2, get_cc = False)
    m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3, get_cc = False)

    summary_df_train.loc[len(summary_df_train)] = [
        storm_indices_train[storm_index],
        *m_eq,
        *m_burton,
        *m_obm,
        *m_ddm1,
        *m_ddm2,
        *m_ddm3,
    ]

summary_df_train.loc[len(summary_df_train)] = ["Mean", *summary_df_train[columns].mean().values]

global_data = pd.concat(storms_train, ignore_index=True)
y_true = global_data["Observed_DST"].values
res_eq = global_data["Pred_DST_Equation"].values
res_burton = global_data["Pred_DST_Burton"].values
res_obm = global_data["Pred_DST_OBM"].values
res_ddm1 = global_data["Pred_DST_DDM1"].values
res_ddm2 = global_data["Pred_DST_DDM2"].values
res_ddm3 = global_data["Pred_DST_DDM3"].values

m_eq = baseline_models.get_all_metrics(y_true, res_eq, get_cc = False)
m_burton = baseline_models.get_all_metrics(y_true, res_burton, get_cc = False)
m_obm = baseline_models.get_all_metrics(y_true, res_obm, get_cc = False)
m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1, get_cc = False)
m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2, get_cc = False)
m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3, get_cc = False)

summary_df_train.loc[len(summary_df_train)] = [
    "Global",
    *m_eq,
    *m_burton,
    *m_obm,
    *m_ddm1,
    *m_ddm2,
    *m_ddm3,
]

display(summary_df_train)

,Storm Index,Equation_RMSE,Equation_MAE,Equation_R2,Equation_BFE,Burton_RMSE,Burton_MAE,Burton_R2,Burton_BFE,OBM_RMSE,...,DDM1_R2,DDM1_BFE,DDM2_RMSE,DDM2_MAE,DDM2_R2,DDM2_BFE,DDM3_RMSE,DDM3_MAE,DDM3_R2,DDM3_BFE
0,1.0,20.213585,17.117328,0.640361,16.822423,29.687764,25.991919,0.224225,26.863936,16.239690,...,0.623787,13.437801,22.568521,19.475600,0.551681,14.829997,23.252721,19.882235,0.524086,19.966095
1,2.0,22.835546,17.477713,-0.337692,19.405386,12.639767,10.815605,0.590163,9.675873,15.425445,...,0.114390,19.256149,18.874640,14.104114,0.086117,18.701587,21.678056,16.169453,-0.205519,21.927198
2,3.0,11.215986,8.793026,0.879061,8.793067,21.754853,17.200253,0.545010,26.594894,13.845367,...,0.769102,20.804152,14.543500,12.109214,0.796658,19.227867,15.653912,12.848885,0.764421,21.447613
3,4.0,17.213986,14.144183,0.778240,19.953851,41.306320,29.321742,-0.276890,39.673857,16.878955,...,0.701980,21.296023,20.949735,17.795732,0.671544,21.903819,20.551589,17.276006,0.683910,22.014097
4,5.0,18.665306,14.714571,0.766902,24.272886,32.265115,22.419636,0.303479,52.045543,18.101877,...,0.706520,27.490935,22.171150,16.574360,0.671115,29.435293,22.597948,16.987666,0.658331,29.757253
5,6.0,15.532397,12.371153,0.645368,16.965733,28.003199,19.706273,-0.152700,33.863118,12.439271,...,0.611709,12.948621,18.066086,15.496299,0.520234,17.225424,17.936579,15.021498,0.527088,17.003694
6,7.0,10.465127,8.609895,0.808674,7.050362,19.740618,16.414759,0.319220,24.766050,11.199831,...,0.683914,17.712507,12.238735,10.768601,0.738327,14.937157,13.896036,12.057634,0.662661,18.509792
7,8.0,12.263987,9.014927,0.847011,13.263640,19.622142,14.975900,0.608359,21.776960,9.682736,...,0.877692,9.832557,10.379121,8.237262,0.890424,8.695989,11.590607,9.317260,0.863351,10.319187
8,9.0,12.576511,9.406606,0.692773,11.592131,17.275970,14.840411,0.420273,13.234930,7.914783,...,0.735905,11.832959,11.275480,8.936104,0.753050,10.333607,12.823987,9.634174,0.680563,11.559362
9,10.0,13.050190,9.008404,0.639154,16.813556,13.522280,11.154802,0.612575,11.519395,10.510103,...,0.653036,19.511731,14.126297,8.265998,0.577191,21.448999,16.740966,8.877678,0.406188,25.359561


In [21]:
display(summary_df_train[['Storm Index', 'Equation_BFE', 'Burton_BFE', 'OBM_BFE', 'DDM1_BFE', 'DDM2_BFE', 'DDM3_BFE']])

,Storm Index,Equation_BFE,Burton_BFE,OBM_BFE,DDM1_BFE,DDM2_BFE,DDM3_BFE
0,1.0,16.822423,26.863936,10.524909,13.437801,14.829997,19.966095
1,2.0,19.405386,9.675873,12.403754,19.256149,18.701587,21.927198
2,3.0,8.793067,26.594894,17.669043,20.804152,19.227867,21.447613
3,4.0,19.953851,39.673857,19.039604,21.296023,21.903819,22.014097
4,5.0,24.272886,52.045543,23.873085,27.490935,29.435293,29.757253
5,6.0,16.965733,33.863118,14.424534,12.948621,17.225424,17.003694
6,7.0,7.050362,24.766050,13.319743,17.712507,14.937157,18.509792
7,8.0,13.263640,21.776960,10.519789,9.832557,8.695989,10.319187
8,9.0,11.592131,13.234930,7.664309,11.832959,10.333607,11.559362
9,10.0,16.813556,11.519395,13.155042,19.511731,21.448999,25.359561


In [22]:
storms = range(54, 74)
parent_folder = 'baseline-comparison-figures'

for storm_number in storms:
    # We use f-strings with double {{ }} to escape the LaTeX braces
    # and single { } for the Python variables.
    latex_code = f"""
\\begin{{figure}}[ht]
    \\centering
    \\includegraphics[width=\\textwidth]{{{parent_folder}/storm_{storm_number}.png}}
    \\caption{{Reconstruction of storm {storm_number} using Eq. 21 against the five baseline models}}\\label{{fig:baseline-comparison-storm-{storm_number}}}
\\end{{figure}}
"""
    print(latex_code)   


\begin{figure}[ht]
    \centering
    \includegraphics[width=\textwidth]{baseline-comparison-figures/storm_54.png}
    \caption{Reconstruction of storm 54 using Eq. 21 against the five baseline models}\label{fig:baseline-comparison-storm-54}
\end{figure}


\begin{figure}[ht]
    \centering
    \includegraphics[width=\textwidth]{baseline-comparison-figures/storm_55.png}
    \caption{Reconstruction of storm 55 using Eq. 21 against the five baseline models}\label{fig:baseline-comparison-storm-55}
\end{figure}


\begin{figure}[ht]
    \centering
    \includegraphics[width=\textwidth]{baseline-comparison-figures/storm_56.png}
    \caption{Reconstruction of storm 56 using Eq. 21 against the five baseline models}\label{fig:baseline-comparison-storm-56}
\end{figure}


\begin{figure}[ht]
    \centering
    \includegraphics[width=\textwidth]{baseline-comparison-figures/storm_57.png}
    \caption{Reconstruction of storm 57 using Eq. 21 against the five baseline models}\label{fig:baseline-compari